<a href="https://colab.research.google.com/github/olblinova/python_for_ds_tasks-is-available/blob/main/%22HW_15_4_%D0%90%D0%BD%D0%B0%D0%BB%D1%96%D0%B7_%D0%90_%D0%92_%D1%82%D0%B5%D1%81%D1%82%D1%96%D0%B2_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Аналіз A/B-тестів

Ви - аналітик даних в ІТ-компанії і до вас надійшла задача проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це - гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку.

У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми хочемо зрозуміти, як це вплинуло на утримання (retention) гравців. Тобто хочемо зрозуміти, чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Колонки в даних наступні:

- `userid` - унікальний номер, який ідентифікує кожного гравця.
- `version` - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- `sum_gamerounds` - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- `retention_1` - чи через 1 день після встановлення гравець повернувся і почав грати?
- `retention_7` - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

1. Для початку, уявімо, що ми тільки плануємо проведення зазначеного А/B-тесту і хочемо зрозуміти, дані про скількох користувачів нам треба зібрати, аби досягнути відчутного ефекту. Відчутним ефектом ми вважатимемо збільшення утримання на 1% після внесення зміни. Обчисліть, скільки користувачів сумарно нам треба аби досягнути такого ефекту, якщо продакт менеджер нам повідомив, що базове утримання є 19%.

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.stats.api as sms
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from math import ceil

%matplotlib inline


In [14]:
df=pd.read_csv('/content/drive/MyDrive/data/cookie_cats.csv')
df.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


In [ ]:
# Гіпотези:
# H0: p₄₀ = p₃₀   (переміщення воріт з 30 на 40 рівень не вплинуло на утримання)
# Ha: p₄₀ ≠ p₃₀   (переміщення воріт вплинуло на утримання)
# Тип тесту: двосторонній z-тест для двох часток

In [15]:
from statsmodels.stats.proportion import proportion_effectsize
from statsmodels.stats.power import NormalIndPower

p_base = 0.19     # базове утримання
p_new = 0.20      # утримання з ефектом +1 в. п.
alpha = 0.05
power = 0.8

effect_size = proportion_effectsize(p_new, p_base)   # ефект Коена h

n_per_group = NormalIndPower().solve_power(
    effect_size=effect_size,
    alpha=alpha,
    power=power,
    ratio=1,                  # групи однакового розміру
    alternative='two-sided'
)

n_per_group = int(np.ceil(n_per_group))
print(f"Ефект h = {effect_size:.4f}")
print(f"На одну групу: {n_per_group}")
print(f"Сумарно: {2 * n_per_group}")

Ефект h = 0.0252
На одну групу: 24638
Сумарно: 49276


2. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [16]:
print(df.groupby('version')['retention_7'].agg(['mean', 'count']))

             mean  count
version                 
gate_30  0.190201  44700
gate_40  0.182000  45489


In [ ]:
# Гіпотези:
# H0: p₃₀ = p₄₀   (утримання на 7-й день однакове в обох версіях)
# Ha: p₃₀ ≠ p₄₀   (утримання на 7-й день відрізняється між версіями)
# Тип тесту: двосторонній z-тест для двох часток

3. Перевірте з допомогою пасуючого варіанту z-тесту, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для варіантів до переміщення воріт і після. Виведіть результат у форматі:

    ```
    z statistic: ...
    p-value: ...
    Довірчий інтервал 95% для групи control: [..., ...]
    Довірчий інтервал 95% для групи treatment: [..., ...]
    ```

    де замість `...` - обчислені значення.
    
    В якості висновку дайте відповідь на два питання:  

      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  


In [17]:
import numpy as np
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

alpha = 0.05

# Кількість гравців, що повернулися на 7-й день, і розмір груп
control = df[df['version'] == 'gate_30']['retention_7']
treatment = df[df['version'] == 'gate_40']['retention_7']

count = np.array([control.sum(), treatment.sum()])
nobs = np.array([len(control), len(treatment)])

# Двосторонній z-тест для двох пропорцій
z_stat, p_value = proportions_ztest(count, nobs, alternative='two-sided')

# 95% довірчі інтервали для кожної групи
ci_control = proportion_confint(count[0], nobs[0], alpha=alpha, method='normal')
ci_treatment = proportion_confint(count[1], nobs[1], alpha=alpha, method='normal')

print(f"z statistic: {z_stat:.4f}")
print(f"p-value: {p_value:.4f}")
print(f"Довірчий інтервал 95% для групи control: [{ci_control[0]:.4f}, {ci_control[1]:.4f}]")
print(f"Довірчий інтервал 95% для групи treatment: [{ci_treatment[0]:.4f}, {ci_treatment[1]:.4f}]")

z statistic: 3.1644
p-value: 0.0016
Довірчий інтервал 95% для групи control: [0.1866, 0.1938]
Довірчий інтервал 95% для групи treatment: [0.1785, 0.1855]


1. Різниця статистично значуща: p-value = 0.0016 < α = 0.05, тому H0
   відхиляємо. Утримання на 7-й день вище у версії gate_30 (19.02%),
   ніж у gate_40 (18.20%), тобто переміщення воріт на 40 рівень
   погіршило утримання приблизно на 0.8 в. п.

2. Довірчі інтервали не перетинаються: [0.1866, 0.1938] для control
   і [0.1785, 0.1855] для treatment. Верхня межа treatment (0.1855)
   менша за нижню межу control (0.1866). Це узгоджується з результатом
   тесту й підтверджує, що різниця між версіями не випадкова. Інтервали не перетинаються, але проміжок між ними дуже малий (менше 0.1 в. п.). Тому ефект є, але невеликий за розміром.

4. Виконайте тест Хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та утриманням гравця на 7ий день після реєстрації.

    - Напишіть, як для цього тесту будуть сформульовані гіпотези.
    - Проведіть обчислення, виведіть p-значення і напишіть висновок за результатами тесту.


Гіпотези:
H0: версія гри і утримання на 7-й день незалежні
    (частка гравців, що повернулися, не залежить від версії)

Ha: версія гри і утримання на 7-й день залежні
    (частка гравців, що повернулися, залежить від версії)

In [18]:
# Таблиця спряженості: версія × retention_7
contingency = pd.crosstab(df['version'], df['retention_7'])
print(contingency)

chi2, p_value, dof, expected = stats.chi2_contingency(contingency)

print(f"chi2 = {chi2:.4f}")
print(f"p-value = {p_value:.4f}")
print(f"ступені свободи = {dof}")

if p_value < alpha:
    print("p-value < α → відхиляємо H0: є залежність між версією і утриманням")
else:
    print("p-value ≥ α → не відхиляємо H0: залежності не виявлено")

retention_7  False  True 
version                  
gate_30      36198   8502
gate_40      37210   8279
chi2 = 9.9591
p-value = 0.0016
ступені свободи = 1
p-value < α → відхиляємо H0: є залежність між версією і утриманням


Висновок: p-value = 0.0016 < α = 0.05, тому H0 відхиляємо. Між версією
гри та утриманням гравців на 7-й день є статистично значуща залежність:
частка гравців, що повернулися, відрізняється між gate_30 і gate_40.
Це узгоджується з результатом z-тесту (z² ≈ 10.01 ≈ χ²).
χ²-тест лише виявляє залежність, а напрямок дають середні: утримання вище в gate_30 (19.02%), ніж у gate_40 (18.20%).